In [9]:
# Diagnostic: show notebook working directory and available data directories
from pathlib import Path
print('CWD:', Path.cwd())
print('\nTop-level folders matching "Pass":')
for p in Path.cwd().iterdir():
    if p.is_dir() and 'pass' in p.name.lower():
        print('-', repr(p.name))

print('\nContents of Pass 3 folder (if present):')
pass3 = None
for p in Path.cwd().iterdir():
    if p.is_dir() and 'pass' in p.name.lower() and '3' in p.name:
        pass3 = p
        break
if pass3:
    for qq in pass3.iterdir():
        print('-', qq.name)
else:
    print('Pass 3 folder not found in CWD')

CWD: /Users/aravshah/Documents/macro engine

Top-level folders matching "Pass":
- 'Pass 2 - Macro State Vector'
- 'Pass 3 - Dataset Work '
- 'Pass 1 - FETCH'

Contents of Pass 3 folder (if present):
- .DS_Store
- F-F_Research_Data_5_Factors_2x3.csv
- Factor_Returns_Final.csv
- loading.ipynb
- F-F_Momentum_Factor.csv


# Compare Macro Scores with Factor Returns (September 2025)

This small utility loads your cleaned factor returns and the macro scores, prompts you to pick one or more factors (Value, Quality, Momentum), and reports which factor performed best in September 2025. It also shows the macro scores for that month and simple correlations for context.


In [ ]:
# Pass 3: Regime-conditioned factor return aggregation (non-interactive)
from pathlib import Path
import pandas as pd
import numpy as np

# Resolve data directories robustly
root = Path.cwd()
pass2_dir = next((p for p in root.iterdir() if p.is_dir() and 'pass' in p.name.lower() and '2' in p.name), None)
pass3_dir = next((p for p in root.iterdir() if p.is_dir() and 'pass' in p.name.lower() and '3' in p.name), None)

if pass2_dir:
    macro_fp = pass2_dir / 'macro_data_scored.csv'
else:
    macro_fp = Path('..') / 'Pass 2 - Macro State Vector' / 'macro_data_scored.csv'

# Use only the canonical file for factor returns
if pass3_dir:
    factors_fp = pass3_dir / 'Factor_Returns_Final.csv'
else:
    factors_fp = Path('.') / 'Factor_Returns_Final.csv'

if not macro_fp.exists():
    raise FileNotFoundError(f"Macro file not found: {macro_fp}")
if not factors_fp.exists():
    dirp = pass3_dir if pass3_dir else Path('.')
    available = [p.name for p in dirp.iterdir()] if dirp.exists() else []
    raise FileNotFoundError(f"Factor file not found. Expected 'Factor_Returns_Final.csv'. Files in {dirp}: {available}")

# Load data
macro_df = pd.read_csv(macro_fp, parse_dates=['DATE'])
factors_df = pd.read_csv(factors_fp, parse_dates=['DATE'])

for df in (macro_df, factors_df):
    df['DATE'] = pd.to_datetime(df['DATE'], errors='coerce')
    df['PERIOD'] = df['DATE'].dt.to_period('M')

# Detect frequency mismatch and align series
# If macro data is quarterly while factors are monthly, expand quarterly macro scores to monthly
# Alternative: aggregate monthly factor returns into quarterly returns (commented)

# Quick freq check: are macro dates only quarter months (Mar/Jun/Sep/Dec)?
macro_months = set(macro_df['DATE'].dt.month.dropna().unique())
quarter_months = {3,6,9,12}
is_macro_quarterly = macro_months.issubset(quarter_months) and len(macro_months) <= 4

if is_macro_quarterly:
    print('Detected quarterly macro data. Expanding to monthly by mapping quarter scores to each month within quarter.')
    # compute quarter period and deduplicate quarter-level scores
    macro_df['QPERIOD'] = macro_df['DATE'].dt.to_period('Q')
    qcols = [c for c in ['Inflation_Score', 'Growth_Score', 'Macro_Score'] if c in macro_df.columns]
    macro_q = macro_df.drop_duplicates('QPERIOD').set_index('QPERIOD')[qcols]

    # build monthly periods covering the same date range as macro quarterly series
    monthly_index = pd.period_range(start=macro_df['DATE'].min().to_period('M'), end=macro_df['DATE'].max().to_period('M'), freq='M')
    monthly_df = pd.DataFrame({'PERIOD': monthly_index})
    monthly_df['QPERIOD'] = monthly_df['PERIOD'].dt.to_period('Q')

    # merge quarter scores onto each month in that quarter
    monthly = monthly_df.merge(macro_q.reset_index(), on='QPERIOD', how='left')
    macro_monthly = monthly[['PERIOD'] + qcols]
    # replace macro_df with the expanded monthly version
    macro_df = macro_monthly.copy()
    print('Macro months after expansion:', macro_df['PERIOD'].nunique())
else:
    # macro_df already monthly (or irregular). Keep only relevant score columns
    qcols = [c for c in ['Inflation_Score', 'Growth_Score', 'Macro_Score'] if c in macro_df.columns]
    macro_df = macro_df[['PERIOD'] + qcols]

# Note: alternative approach is to aggregate factor returns to quarters using compounded returns:
# factors_q = factors_df.set_index('DATE').resample('Q').apply(lambda s: (1+s).prod()-1)
# If you prefer that approach, set alignment_method = 'aggregate_factors' and implement accordingly.

# Required factor columns
factor_cols = [c for c in ['Value', 'Quality', 'Momentum'] if c in factors_df.columns]
if not factor_cols:
    raise ValueError('No factor return columns found in Factor_Returns_Final. Expected one or more of: Value, Quality, Momentum')

# Merge once (monthly periods) - macro_df now has monthly PERIOD if expanded
merged = pd.merge(
    factors_df[['PERIOD'] + factor_cols],
    macro_df[['PERIOD'] + qcols],
    on='PERIOD',
    how='left'
)
merged = merged.sort_values('PERIOD').reset_index(drop=True)
if merged.empty:
    raise ValueError('Merged factor/macro dataset is empty — check inputs.')

# --- Diagnostics: check score ranges and missingness ---
print('Merged shape:', merged.shape)
print('Unique periods:', merged['PERIOD'].nunique())
print('\nSample rows:')
display(merged.head(10))

score_cols = qcols
for c in score_cols:
    ser = merged[c]
    print(f"\n{c}: non-null={ser.notna().sum()}, null={ser.isna().sum()}")
    try:
        print(ser.describe())
        print('Quantiles:', ser.quantile([0.0, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 1.0]).to_dict())
    except Exception:
        pass

# ------- Pass 3 canonical operations -------
# Step 1 — Define regimes (parameterizable thresholds)
# These thresholds are starting recommendations; adjust as needed later.
thresholds = {
    'inflation_high': 1.0,   # Inflation_Score >= +1
    'growth_weak': -1.0,     # Growth_Score <= -1
    'liquidity_tight': -1.0, # Macro_Score <= -1
}

# Regime definitions: functions that accept dataframe and return boolean mask
regimes = {
    'High_Inflation': lambda df: df['Inflation_Score'].ge(thresholds['inflation_high']),
    'Weak_Growth':     lambda df: df['Growth_Score'].le(thresholds['growth_weak']),
    'Tight_Liquidity': lambda df: df['Macro_Score'].le(thresholds['liquidity_tight']),
}

# Check how many observations each absolute-threshold regime would select
abs_counts = {name: int(cond(merged).sum()) for name, cond in regimes.items()}
print('\nCounts using absolute thresholds:', abs_counts)

# If regimes are empty or very small, use a quantile-based fallback
min_count_thresh = max(5, int(0.05 * len(merged)))  # at least 5 or 5% of data
need_fallback = any(cnt < min_count_thresh for cnt in abs_counts.values())

if need_fallback:
    print('\nOne or more regimes have too few observations. Computing quantile-based fallback thresholds...')
    q_high = 0.75
    q_low = 0.25
    q_thresholds = {}
    if 'Inflation_Score' in merged.columns:
        q_thresholds['inflation_q'] = merged['Inflation_Score'].dropna().quantile(q_high)
    if 'Growth_Score' in merged.columns:
        q_thresholds['growth_q'] = merged['Growth_Score'].dropna().quantile(q_low)
    if 'Macro_Score' in merged.columns:
        q_thresholds['macro_q'] = merged['Macro_Score'].dropna().quantile(q_low)
    print('Quantile thresholds (inflation>=75%, growth<=25%, macro<=25):', q_thresholds)

    regimes_q = {
        'High_Inflation': lambda df: df['Inflation_Score'].ge(q_thresholds.get('inflation_q', np.nan)),
        'Weak_Growth':     lambda df: df['Growth_Score'].le(q_thresholds.get('growth_q', np.nan)),
        'Tight_Liquidity': lambda df: df['Macro_Score'].le(q_thresholds.get('macro_q', np.nan)),
    }
    q_counts = {name: int(cond(merged).sum()) for name, cond in regimes_q.items()}
    print('Counts using quantile thresholds:', q_counts)

    # choose regimes_q if it has larger counts overall
    if sum(q_counts.values()) >= sum(abs_counts.values()):
        print('Using quantile-based regimes for summary (more observations).')
        regimes_to_use = regimes_q
    else:
        print('Quantile fallback did not increase observations significantly; keeping absolute thresholds.')
        regimes_to_use = regimes
else:
    regimes_to_use = regimes

# Mark which regime definition we are using ('absolute' or 'quantile')
if 'regimes_q' in locals() and regimes_to_use is regimes_q:
    regime_definition = 'quantile'
else:
    regime_definition = 'absolute'

# Step 2 & 3 — Slice history and compute regime-conditioned averages

def compute_regime_stats(df: pd.DataFrame, regimes: dict, factor_cols: list, regime_definition: str) -> pd.DataFrame:
    """Return a DataFrame summarizing mean returns, std, count and ranking per regime.

    Adds `coverage_pct` = n / len(df) and `regime_definition` ("absolute" or "quantile") to each row.
    """
    rows = []
    total = len(df)
    for name, cond in regimes.items():
        mask = cond(df) & df[factor_cols].notna().any(axis=1)
        subset = df.loc[mask, factor_cols]
        n = subset.shape[0]
        coverage = n / total if total > 0 else np.nan
        if n == 0:
            rows.append({
                'regime': name,
                'n_periods': 0,
                'coverage_pct': coverage,
                'regime_definition': regime_definition,
                'note': 'no observations'
            })
            continue
        means = subset.mean()
        stds = subset.std()
        sem = stds / np.sqrt(n)
        # rank factors by mean (descending)
        rank = means.sort_values(ascending=False).index.tolist()
        best = rank[0]
        row = {
            'regime': name,
            'n_periods': n,
            'coverage_pct': coverage,
            'regime_definition': regime_definition,
            'best_factor': best,
            'best_mean': means[best],
        }
        # include each factor mean and sem for transparency
        for f in factor_cols:
            row[f'{f}_mean'] = means.get(f, np.nan)
            row[f'{f}_sem'] = sem.get(f, np.nan)
        rows.append(row)
    return pd.DataFrame(rows)

regime_summary = compute_regime_stats(merged, regimes_to_use, factor_cols, regime_definition)

# Step 4 — Save and present the conditioning table for Pass 3
out_fp = (pass3_dir if pass3_dir else Path('.')) / 'regime_factor_return_summary.csv'
out_fp.parent.mkdir(parents=True, exist_ok=True)
regime_summary.to_csv(out_fp, index=False)

# Provide a concise programmatic result and a short printed summary (no user prompts)
print(f"Saved regime summary to: {out_fp}")
print(f"Regime definition used: {regime_definition}")
print('\nRegime summary (top rows):')
print(regime_summary[['regime', 'n_periods', 'coverage_pct', 'regime_definition', 'best_factor', 'best_mean']])

# Expose useful artifacts for downstream Pass 4 or further analysis
PASS3_RESULTS = {
    'merged': merged,
    'regime_summary': regime_summary,
    'out_fp': out_fp,
}

# Notes:
# - Pass 3 is intentionally non-interactive: it computes the conditioning table for historical analysis.
# - If absolute thresholds select too few observations, the code falls back to quantile-based thresholds to ensure regimes are populated.
# - Alternative: aggregate monthly factor returns to quarters instead of expanding macro series; see comments above.
# - For Pass 4 (user entry analysis), write a separate utility that uses PASS3_RESULTS['regime_summary'] to map current macro state to recommended factors.


Merged shape: (738, 7)
Unique periods: 738

Sample rows:


,PERIOD,Value,Quality,Momentum,Inflation_Score,Growth_Score,Macro_Score
0,1963-07,-0.0097,0.0068,0.0101,NaN,NaN,NaN
1,1963-08,0.0180,0.0036,0.0100,NaN,NaN,NaN
2,1963-09,0.0013,-0.0071,0.0012,NaN,NaN,NaN
3,1963-10,-0.0010,0.0280,0.0313,NaN,NaN,NaN
4,1963-11,0.0175,-0.0051,-0.0078,NaN,NaN,NaN
5,1963-12,-0.0002,0.0003,0.0184,NaN,NaN,NaN
6,1964-01,0.0148,0.0017,0.0102,NaN,NaN,NaN
7,1964-02,0.0281,-0.0005,0.0033,NaN,NaN,NaN
8,1964-03,0.0340,-0.0221,0.0069,NaN,NaN,NaN
9,1964-04,-0.0067,-0.0127,-0.0064,NaN,NaN,NaN



Inflation_Score: non-null=51, null=687
count    51.000000
mean     -0.138357
std       0.544631
min      -1.280481
25%      -0.601868
50%       0.000000
75%       0.296261
max       0.690679
Name: Inflation_Score, dtype: float64
Quantiles: {0.0: -1.2804814262229818, 0.01: -1.2622859557549984, 0.05: -1.1047973721657478, 0.25: -0.6018682417161392, 0.5: 0.0, 0.75: 0.2962609786972733, 0.95: 0.5030107387689515, 0.99: 0.6432409002680735, 1.0: 0.6906785652683358}

Growth_Score: non-null=51, null=687
count    51.000000
mean      0.344028
std       1.051014
min      -2.206486
25%      -0.075308
50%       0.500000
75%       0.917873
max       2.229790
Name: Growth_Score, dtype: float64
Quantiles: {0.0: -2.206485887027883, 0.01: -2.1304166534646636, 0.05: -1.8168919104437422, 0.25: -0.07530773046088785, 0.5: 0.5, 0.75: 0.9178729072896961, 0.95: 2.0135584422421275, 0.99: 2.15718290944029, 1.0: 2.2297900230032006}

Macro_Score: non-null=51, null=687
count    51.000000
mean      0.249929
std       

In [6]:
# Quick tests/validation for Pass 3 outputs (lightweight)
from pathlib import Path
import pandas as pd

out_fp = PASS3_RESULTS.get('out_fp') if 'PASS3_RESULTS' in globals() else None
regime_summary = PASS3_RESULTS.get('regime_summary') if 'PASS3_RESULTS' in globals() else None

# Basic checks (these are non-blocking; user should run the cell and inspect results)
if out_fp is None or not Path(out_fp).exists():
    print('WARNING: regime summary CSV not found at expected path:', out_fp)
else:
    print('Regime summary CSV exists at:', out_fp)

if regime_summary is None or regime_summary.empty:
    print('WARNING: regime_summary is empty or not found. Did Pass 3 run correctly?')
else:
    print('\nRegime summary (full):')
    display(regime_summary)
    # show ranking table
    print('\nRanking by regime:')
    print(regime_summary[['regime','n_periods','best_factor','best_mean']].to_string(index=False))

# If these checks look good, mark the todo completed in the todo list (manual step).

Regime summary CSV exists at: /Users/aravshah/Documents/macro engine/Pass 3 - Dataset Work /regime_factor_return_summary.csv

Regime summary (full):


,regime,n_periods,best_factor,best_mean,Value_mean,Value_sem,Quality_mean,Quality_sem,Momentum_mean,Momentum_sem
0,High_Inflation,13,Value,0.007908,0.007908,0.009072,0.007138,0.005277,-0.035108,0.029721
1,Weak_Growth,13,Value,0.009154,0.009154,0.010038,0.003338,0.007693,-0.026554,0.030237
2,Tight_Liquidity,13,Quality,0.008362,0.001800,0.009735,0.008362,0.004767,0.002638,0.011492



Ranking by regime:
         regime  n_periods best_factor  best_mean
 High_Inflation         13       Value   0.007908
    Weak_Growth         13       Value   0.009154
Tight_Liquidity         13     Quality   0.008362


### Pass 3 design and rationale
#
# Pass 3 must be a historical conditioning step (non-interactive). It should:
# 1) Define regimes (e.g., High Inflation, Weak Growth, Tight Liquidity)
# 2) Slice history into those regimes
# 3) Compute average factor returns per regime
# 4) Rank factors within each regime and export a summary CSV
#
# This cell implements those steps and produces `Pass 3 - Dataset Work/regime_factor_return_summary.csv`.
